<a href="https://colab.research.google.com/github/rallyfranky/my-first-repo/blob/main/newspaper_budget_optimize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pulp
import pandas as pd
import sys
from __future__ import annotations

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/sites.csv')

,拠点番号,拠点都道府県,数値1,数値2
0,S00001,茨城県,714000,370
1,S00002,千葉県,485000,261
2,S00003,大阪府,1940000,969
3,S00004,山口県,771000,246
4,S00005,千葉県,446000,216


In [4]:
df = df.rename(columns={'拠点番号': 'branch_no',
                   '拠点都道府県':'pref',
                   '数値1': 'budget',
                   '数値2': 'kpi'})

In [25]:
def news_optimize(df: pd.DataFrame, budget: float,min_pref: float):
  df = df.reset_index(drop=True)
  idx = list(df.index)
  b = df['budget'].to_dict()
  k = df['kpi'].to_dict()
  prob = pulp.LpProblem('newspaper_buying', pulp.LpMaximize)
  #決定変数、x={0,1}拠点採択のバイナリ
  x = pulp.LpVariable.dicts('x', idx, cat=pulp.LpBinary)
  #目的変数（KPI最大化）
  prob += pulp.lpSum(x[i] * k[i] for i in idx), "total_kpi"
  #予算制約
  prob += pulp.lpSum(x[i] * b[i] for i in idx) <= budget, "total_budget"
  #地域制約
  for pref, member in df.groupby('pref').groups.items():
    prob += (
        pulp.lpSum(x[i] for i in member) >= 1,
        f"cover_{pref}",
    )
  prob.solve()

  print("Status:", pulp.LpStatus[prob.status])
  selected_branches = [i for i in idx if x[i].varValue == 1]
  total_kpi = pulp.value(prob.objective)
  total_budget_spent = sum(b[i] for i in selected_branches)
  return selected_branches, total_kpi, total_budget_spent

In [26]:
news_optimize(df, 65000000, 1)

Status: Optimal


([11,
  12,
  14,
  25,
  32,
  51,
  57,
  77,
  95,
  98,
  176,
  201,
  211,
  213,
  218,
  225,
  233,
  234,
  246,
  259,
  262,
  269,
  277,
  291,
  295,
  296,
  298,
  299,
  319,
  324,
  340,
  348,
  375,
  379,
  386,
  389,
  395,
  397,
  399,
  401,
  402,
  410,
  419,
  440,
  451,
  469,
  482,
  526,
  531,
  538,
  568,
  578,
  582,
  592,
  608,
  612,
  615,
  625,
  637,
  641,
  657,
  661,
  677,
  691,
  699,
  705,
  727,
  745,
  754,
  755,
  767,
  782,
  797,
  803,
  807,
  810,
  830,
  867,
  884,
  897,
  902,
  930,
  935,
  938,
  941,
  977,
  983,
  988,
  997,
  998,
  1010,
  1014,
  1016,
  1021,
  1049,
  1067,
  1089,
  1101,
  1113,
  1124,
  1153,
  1163,
  1167,
  1174,
  1175,
  1183,
  1185,
  1199],
 64716.0,
 64995000)